In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [3]:
# ==========================================
# 1. LOAD FEATURES FROM CSV
# ==========================================
import pandas as pd
import re

FEATURES_CSV = "features_extracted.csv"

df = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in df.columns if re.fullmatch(r"f\d+", c)]

X_features = df[feature_cols].to_numpy(dtype=np.float32)
y_labels = df["label"].to_numpy()

print(f"Loaded {len(df)} rows, {len(feature_cols)} features")
print(f"Real: {(y_labels == 0).sum()}, AI: {(y_labels == 1).sum()}")

Loaded 155015 rows, 51 features
Real: 50000, AI: 105015


In [4]:
# ==========================================
# 3. DATA SPLIT AND NORMALIZATION
# ==========================================
USING_MODEL = "RF"  # "SVM"

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# ==========================================
# 4. TRAINING
# ==========================================

print("Bắt đầu huấn luyện mô hình...")

if USING_MODEL == "RF":
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) 
else:
    model = SVC(kernel='rbf', C=1.0, random_state=42)

model.fit(X_train_scaled, y_train)
print("Huấn luyện xong!\n")

# 5. Đánh giá trên tập Test
print("--- KẾT QUẢ ĐÁNH GIÁ (TEST SET) ---")
y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Báo cáo phân loại chi tiết:")
print(classification_report(y_test, y_pred, target_names=["Real Art (0)", "AI Art (1)"]))

# (Tùy chọn) In thêm Ma trận nhầm lẫn để xem máy hay đoán nhầm Real thành AI hay ngược lại
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Bắt đầu huấn luyện mô hình...
Huấn luyện xong!

--- KẾT QUẢ ĐÁNH GIÁ (TEST SET) ---
Accuracy: 97.41%

Báo cáo phân loại chi tiết:
              precision    recall  f1-score   support

Real Art (0)       0.97      0.95      0.96     10000
  AI Art (1)       0.98      0.98      0.98     21003

    accuracy                           0.97     31003
   macro avg       0.97      0.97      0.97     31003
weighted avg       0.97      0.97      0.97     31003

Confusion Matrix:
[[ 9514   486]
 [  318 20685]]


In [6]:
# ==========================================
# 6. SAVE MODEL + SCALER
# ==========================================
import joblib

MODEL_PATH = f"model_{USING_MODEL.lower()}.joblib"
SCALER_PATH = "scaler.joblib"

joblib.dump(model, MODEL_PATH, compress=3)
joblib.dump(scaler, SCALER_PATH, compress=3)

print(f"Saved model to {MODEL_PATH}")
print(f"Saved scaler to {SCALER_PATH}")

Saved model to model_rf.joblib
Saved scaler to scaler.joblib
